In [3]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urlunparse
from urllib.robotparser import RobotFileParser
from concurrent.futures import ThreadPoolExecutor
from threading import Lock
import requests

import heapq
import time
import math
import json
import ast

## Parse content helper methods

In [7]:
def getResponse(url):
    # Fetch the webpage content
    response = requests.get(url, timeout=2)

    return response

In [8]:
# Check if page is valid and in English
def validPage(response):
    headers = response.headers
    lang = headers["content-language"] if "content-language" in headers else "en"
    contentType = headers["content-type"].split(";") if "content-type" in headers else "text/html"

    return lang == "en" and contentType[0] and contentType[0] == "text/html"

In [9]:
def getHtmlContent(response):
    htmlContent = response.text

    # Parse the HTML content
    return BeautifulSoup(htmlContent, 'html.parser')

## Get tag content

In [10]:
def getTitle(soup):
    title = soup.find('title')
    return title.text if title else ""

In [11]:
def getText(soup):
    text = ""

    # Find all elements of p tag
    paras = soup.find_all('p')

    for para in paras:
        text += para.text

    return text

In [12]:
def getFileContent(url, title, text):
    return f"<DOC>\n<DOCNO>{url}<\DOCNO>\n<HEAD>{title}<\HEAD>\n<TEXT>{text}<\TEXT><\DOC>\n"

def writePqIntoFile(fileName, content):
    with open(fileName, "w",encoding='utf-8') as f:
        for item in content:
            f.write(str(item)+"\n")

def writeIntoFile(fileName, content):
    with open(fileName, "a",encoding='utf-8') as f:
        f.write(content)

In [13]:
def verify_robot_parse(url):
    robot_parser=RobotFileParser()

    robot_parser.set_url(f"{url}+/robots.txt")

    robot_parser.read()
    user_agent="*"

    return robot_parser.can_fetch(user_agent, url)

## URL Canonicalization

In [14]:
def canonicalizeURL(baseUrl, url):
    ## Parses the URL
    parsedUrl = urlparse(url)

    ## 1. Converts to lowercase
    scheme = parsedUrl.scheme.lower()
    netloc = parsedUrl.netloc.lower()

    ## 2. Remove port
    if parsedUrl.port:
        netloc = parsedUrl.hostname

    ## 4. Remove the fragment
    fragment = ""

    ## 5. Remove duplicate //
    path = parsedUrl.path.replace('//', '/')

    ## 3. Get absolute path
    if not scheme:
        parsedBaseUrl = urlparse(baseUrl)
        scheme = parsedBaseUrl.scheme.lower()
        netloc = parsedBaseUrl.netloc.lower()
        path = url

    return urlunparse((scheme, netloc, path, parsedUrl.params, parsedUrl.query, fragment))

In [15]:
def traverseLinks(baseUrl, soup, waveNumber,inlinks,outlinks,pq,inlinksCounter):
    # Find all elements of anchor tag
    links = soup.find_all('a', href=True)

    if baseUrl not in outlinks:
        outlinks[baseUrl] = set()

    for link in links:
        hrefLink = link.get("href")
        cl = canonicalizeURL(baseUrl, hrefLink)

        if cl not in inlinks:
            inlinks[cl] = set()
            inlinksCounter[cl] = 0

        ## from baseUrl, link is going out
        outlinks[baseUrl].add(cl)

        ## from link, baseUrl is coming in
        inlinks[cl].add(baseUrl)
        inlinksCounter[cl] += 1

    if pq.getSize() < 30000:
        for cl in inlinksCounter:
            pq.push(cl, inlinksCounter[cl], waveNumber + 1)

## Setting up priority queue

In [16]:
class PriorityQueue:
    def __init__(self):
        self.KEYWORDS=['accident','chernobyl','kyshtym','disaster','fukushima','nuclear','cancer','radioactive','environment','radiation','contamination','incident','evacuation','gov','org','long-term','safety']
        self._queue = []

    # min heap is maintained by default
    def push(self, link, waveNumber, inLinkCount):
        keyword_hits=0
        for keyword in self.KEYWORDS:
            if keyword in link:
                keyword_hits+=1
        keyword_score=math.exp(-keyword_hits)
        inlink_score=math.exp(-inLinkCount)
        curr_score=keyword_score+inlink_score
        heapq.heappush(self._queue, (waveNumber, curr_score, link))

    def pop(self):
        waveNumber,inLinkCount, link = heapq.heappop(self._queue)
        return waveNumber, link

    def getSize(self):
        return len(self._queue)

    def loadQueue(self, queue):
        self._queue = queue

    def getQueue(self):
        return self._queue

In [18]:
def writeLinksToFile(linksCount,thread_id,inlinks,outlinks):
    inlinks_data={key : list(value) for key,value in inlinks.items()}
    outlinks_data={key : list(value) for key,value in outlinks.items()}
    with open(f"./IR/links/inlinks_{linksCount}_thread_{thread_id}.json",'w') as inlinks_file:
        json.dump(inlinks_data,inlinks_file,indent=2)
    with open(f"./IR/links/outlinks_{linksCount}_thread_{thread_id}.json",'w') as outlinks_file:
        json.dump(outlinks_data,outlinks_file,indent=2)

## Loading priority queue

In [19]:
def getPq(filename):
    # Open the file in read mode
    with open(filename, 'r') as file:
        # Read all lines into a list
        contentPq = file.readlines()

    pq = PriorityQueue()

    def loadPq(contentPq):
        queue = []

        for content in contentPq:
            queue.append(ast.literal_eval(content))

        pq.loadQueue(queue)

    loadPq(contentPq)

    return pq

## Loading inlinks and outlinks

In [20]:
def getInlinks(filename):
    with open(filename, 'r') as file:
        inlinks = json.load(file)

    inlinksCounter = {}

    for link in inlinks:
        inlinksCounter[link] = len(inlinks[link])

    for link in inlinks:
        inlinks[link] = set(inlinks[link])

    return inlinks, inlinksCounter

def getOutLinks(filename):
    with open(filename, 'r') as file:
        outlinks = json.load(file)

    for link in outlinks:
        outlinks[link] = set(outlinks[link])

    return outlinks

## Document processing

In [22]:
visited_urls= {}
lock = Lock()

def crawl_seed_URL(baseUrl, thread_id):
    global visited_urls

    if baseUrl:
        print("Base url is there", baseUrl)
        inlinks = {}
        inlinksCounter = {}
        outlinks = {}

        pq = PriorityQueue()
        pq.push(baseUrl, 0, 0)
    else:
        inlinks = inlinks
        inlinksCounter = inlinksCounter
        outlinks = outlinks

        pq = pq


    now = time.time()

    linksCount = 1
    data=""
    file_id=1

    # Set to 30,000 / total_seed_urls you have i.e. 30,000 / 5 = 6000
    while linksCount <= 6000:
        time.sleep(1)
        waveNumber, currentUrl = pq.pop()

        if currentUrl not in visited_urls:
            # To verify the links and create more seeds to improve speed
            visited_urls[currentUrl] = 1
        else:
            continue

        parsedUrl = urlparse(currentUrl)
        hostname = parsedUrl.netloc.lower().split('.')

        if 'ru' in hostname[-1] or 'de' in hostname[-1]:
            continue

        try:
            if linksCount % 1000 == 0:
              print(f"On {linksCount} for thread {thread_id}")

            response = getResponse(currentUrl)

            ## if current page is non-html or not in english or does not match robots.txt requirements
            if not validPage(response) or not verify_robot_parse(currentUrl):
                continue

            soup = getHtmlContent(response)

            title = getTitle(soup)
            text = getText(soup)

            traverseLinks(currentUrl, soup, waveNumber,inlinks,outlinks,pq,inlinksCounter)
            data+=getFileContent(currentUrl,title,text)

            # Decide the number to which we want to write to file, default value can be decided as 1000
            if linksCount % 1000 == 0:
                print(f"Writing files to webpage_data_{file_id} by thread id :{thread_id}.txt")
                writeIntoFile(f"./IR/webpage/webpage_data_{file_id}_thread_{thread_id}.txt", data)
                file_id+=1
                data=""

                writePqIntoFile(f"./IR/pq/priority_queue{linksCount}_thread_{thread_id}.txt", pq.getQueue())
                writeLinksToFile(linksCount,thread_id,inlinks,outlinks)


            linksCount += 1
        except Exception as e:
            pass

    print(f'Time in {time.time() - now} seconds')

In [23]:
"""
Each seed will initiate a different thread, and file_id is the id of the file where the results will be stored, set the file_id unique enough so that multiple threads don't write onto the same file
"""
seed_list=[
    "http://www.world-nuclear.org/info/Safety-and-Security/Safety-of-Plants/Fukushima-Accident/",
    "http://en.wikipedia.org/wiki/Fukushima_Daiichi_nuclear_disaster",
    "https://thebulletin.org/2019/11/an-update-from-fukushima-and-the-challenges-that-remain-there/",
    "https://www.iaea.org/newscenter/focus/fukushima/status-update",
    "http://nuclearsafety.gc.ca/eng/nuclear-substances/exposure-device-operators/becoming-eligible-for-EDO-certification/#VocationalTrainingCourse"
]

In [24]:
with ThreadPoolExecutor(max_workers=5) as thread_executor:
    for thread_id,baseUrl in enumerate(seed_list):
        thread_executor.submit(crawl_seed_URL,baseUrl,thread_id+1)

Base url is there http://en.wikipedia.org/wiki/Chernobyl_disaster
Base url is there http://www.world-nuclear.org/info/Safety-and-Security/Safety-of-Plants/Chernobyl-Accident/
Base url is there http://en.wikipedia.org/wiki/Lists_of_nuclear_disasters_and_radioactive_incidents
Base url is there http://www.world-nuclear.org/info/Safety-and-Security/Safety-of-Plants/Chernobyl-Accident/
Base url is there http://www.world-nuclear.org/information-library/safety-and-security/radiation-and-health.aspx
Base url is there http://bellona.org/news/nuclear-issues/accidents-and-incidents/2006-10-reflections-of-a-chernobyl-liquidator-the-way-it-was-and-the-way-it-will-be
On 500 for thread 3
Writing files to webpage_data_1 by thread id :3.txt
On 500 for thread 1
Writing files to webpage_data_1 by thread id :1.txt
On 500 for thread 6
Writing files to webpage_data_1 by thread id :6.txt
On 500 for thread 2
Writing files to webpage_data_1 by thread id :2.txt
On 1000 for thread 3
Writing files to webpage_data